# STAC AI Workshop · NYC · 21 October 2026

## 07 · Check authority at commit time

**Matthew Dixon, PhD, FRM · Participant workbook**

### Learning objectives
1. Distinguish reasoning-time approval from current permission
2. Enforce state, evidence and authority predicates before dispatch
3. Verify duplicate prevention and auditable decisions
4. State assumptions required for an operational guarantee

Run the cells in order. Python 3.10+ and Jupyter are sufficient. All default exercises run offline. The figures and documents are synthetic unless a cell explicitly identifies recorded model outputs. No financial action is executed.

**Keep the companion `workshop_offline.py` and `data/` folder with these notebooks.**

In [1]:
from pathlib import Path
import sys
notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)
if notebook_dir is None: raise RuntimeError("Extract the complete workshop ZIP and open this notebook inside its notebooks folder.")
if str(notebook_dir) not in sys.path: sys.path.insert(0,str(notebook_dir))
from workshop_offline import *
print("Offline workbook ready: no API key, Node.js, GPU or model download.")

Offline workbook ready: no API key, Node.js, GPU or model download.


## Class Exercise · prepare, change state, commit
This matches **Web Section 7**. The stateful controller holds a proposal, a simulated grant, a holdings version and pending orders. It checks eight predicates **before** adding a mock order.

Everything is in memory. Grants and reviewer approval are supplied fixtures, not authenticated identities. No external broker is called.

In [2]:
data=load_fixture("commit-gate.json")
ctl=CommitController(data["initial_state"])
ctl.prepare(5000)
first=ctl.commit()
show_table([{"Predicate":k,"Result":"PASS" if v else "FAIL"} for k,v in first["checks"].items()])
print("Commit:",first["decision"],"| Mock orders:",len(ctl.orders))
assert first["dispatched"] and len(ctl.orders)==1
second=ctl.commit()
assert second["decision"]=="BLOCKED" and len(ctl.orders)==1
print("Second commit:",second["decision"],"| Duplicate dispatch prevented.")

Predicate,Result
proposal_present,PASS
snapshot_current,PASS
pending_orders_current,PASS
grant_current,PASS
within_limit,PASS
evidence_valid,PASS
reviewer_approved,PASS
not_already_committed,PASS


Commit: PERMITTED | Mock orders: 1
Second commit: BLOCKED | Duplicate dispatch prevented.


## Change the state after preparation
Reset, prepare, then revoke the grant. Earlier approval cannot override current revoked authority.

In [3]:
ctl=CommitController(data["initial_state"])
ctl.prepare();ctl.change("grant");result=ctl.commit()
show_table([{"Predicate":k,"Result":"PASS" if v else "FAIL"} for k,v in result["checks"].items()])
assert not result["dispatched"] and ctl.orders==[]
show_table([{"Sequence":e["sequence"],"Event":e["event"],"Detail":e.get("kind",e.get("decision",e.get("proposal",{}).get("id","")))} for e in ctl.audit])

Predicate,Result
proposal_present,PASS
snapshot_current,PASS
pending_orders_current,PASS
grant_current,FAIL
within_limit,PASS
evidence_valid,PASS
reviewer_approved,PASS
not_already_committed,PASS


Sequence,Event,Detail
1,PREPARED,proposal-1
2,STATE_CHANGED,grant
3,COMMIT_CHECK,BLOCKED


## Test each state transition
Choose `snapshot`, `pending`, `grant`, `evidence`, `approval` or `unchanged`. The proposed action stays the same while current permissions and preconditions may change.

In [4]:
references={r["id"]:r for r in load_fixture("section567-reference.json")["commit"]}
rows=[]
for scenario in data["scenarios"]:
    ctl=CommitController(data["initial_state"]);ctl.prepare();ctl.change(scenario["id"])
    first=ctl.commit();second=ctl.commit()
    ref=references[scenario["id"]]
    assert first==ref["first"] and second==ref["second"] and ctl.orders==ref["orders"] and ctl.audit==ref["audit"]
    assert first["decision"]==scenario["expected"]
    rows.append({"Change":scenario["label"],"Decision":first["decision"],"Failed checks":", ".join(k for k,v in first["checks"].items() if not v),"Orders":len(ctl.orders)})
show_table(rows)
ctl=CommitController(data["initial_state"])
assert ctl.commit()["decision"]=="BLOCKED"  # no proposal
ctl.prepare(10001)
assert ctl.commit()["decision"]=="BLOCKED"  # exceeds delegated limit
print("PASS: all 6 transitions, duplicate dispatch, missing proposal and over-limit checks.")

Change,Decision,Failed checks,Orders
No state change,PERMITTED,,1
Update holdings snapshot,BLOCKED,snapshot_current,0
Add a pending order,BLOCKED,pending_orders_current,0
Revoke the grant,BLOCKED,grant_current,0
Invalidate required evidence,BLOCKED,evidence_valid,0
Withdraw reviewer approval,BLOCKED,reviewer_approved,0


PASS: all 6 transitions, duplicate dispatch, missing proposal and over-limit checks.


## Governance case · Yankees versus Patriots

**Which team is better?** Baseball and American football have no automatic shared measure of “better”. For this hypothetical exercise, the business owner defines a sponsorship comparison memo for a specified audience, market and period.

Follow Web Section 7: unclear objective → one fan poll → mismatched measures → reviewed evidence → current approval.

- **R1 / O1:** Business owner defines the objective.
- **R2 / E1–E2:** Data steward and survey specialist review coverage and methods.
- **R3 / E → M → C:** Statistician and commercial expert validate comparable measures and a bounded claim.
- **R4 / A1:** Named approver authorizes memo release after material checks pass.

These are supplied teaching states, not actual team data, automatic bias discovery, or authenticated approvals. Passing allows a scoped memo, not an overall winner or spending authority.


In [ ]:
# Independently evaluate the same declared policy as the web exercise.
sports_reference = load_fixture("sports-governance.json")
def sports_decision(s):
    if s.get("objective") is not True:
        return "CLARIFY"
    if s.get("coverage") is not True or s.get("comparable") is not True:
        return "ESCALATE / WITHHOLD RANKING"
    if s.get("approval") is not True:
        return "AWAIT APPROVAL"
    return "RELEASE SCOPED MEMO"

for scenario in sports_reference:
    result = sports_decision(scenario["input"])
    assert result == scenario["expected"]["decision"]
    print(scenario["input"]["label"], "→", result)

# Approval cannot compensate for missing definition, evidence or inference.
for requirement in ("objective", "coverage", "comparable"):
    invalid = dict(sports_reference[-1]["input"], **{requirement: False})
    assert sports_decision(invalid) != "RELEASE SCOPED MEMO"
print("PASS: all five web states match; approval cannot override material gaps.")


## Fixed predicates versus sequential decisions
These hard constraints need fixed predicates: current state, valid evidence, active grant, limit and reviewer approval. An MDP/POMDP may help choose whether to wait, investigate or request review when actions change information and consequences. It cannot authorize an otherwise forbidden action.

## What the result guarantees—and does not
The tested mock dispatcher receives an order only when all current checks pass. Production requires authenticated grants, atomic check-and-dispatch, durable idempotency, and no bypass route. Example tests do not constitute a proof of a distributed execution system.

## Apply to your workflow
| Requirement | Owner | Fail action | Record |
|---|---|---|---|
| Eligible source version | Data steward | Hold / replace evidence | Snapshot + cutoff |
| Supported claim | Domain reviewer | Resolve / escalate | Failed dependency |
| Current holdings and orders | Trading operations | Recompute proposal | State versions |
| Active delegated grant | Accountable approver | Block | Grant version + audit |

**Learning check:** can a useful model recommendation bypass a revoked grant? **No.**

## Main takeaway
The evidence graph connects prerequisites; the controller enforces the action boundary. Approval at reasoning time is not permission at execution time.

**Exit ticket:** explain one result, the evidence for it, and a limit on the conclusion.